# 0_DataDownloader_R
Here, we download climate data to ensure that our values are up-to-date. The file climate.csv.gz currently only contains values until the Week April 20-26, 2025. We use the Mosqlimate API here to download all the recent values then.

In [ ]:
library(tidyverse)
library(httr2)

In [ ]:
base_dir = getwd()
raw_data_dir = file.path(base_dir, "data_sprint_2025")
geo_df = read_csv(file.path(raw_data_dir, "map_regional_health.csv")) #Information on geocodes and admin hierarchy

In [ ]:
date_start = "2025-01-01"
date_end = "2025-07-05"
page = 1
per_page = 200

api = "https://api.mosqlimate.org/api/datastore/climate/"
pagination = paste0("?page=", page, "&per_page=", per_page, "&")
filters = paste0("start=", date_start, "&end=", date_end)
api_key = "hadrianang:cb1a6931-0b81-4fa7-bf78-380dc889eef9"

In [ ]:
uf_list = geo_df %>% pull(uf) %>% unique

In [ ]:
curr_uf = uf_list[1]
curr_geo = geo_df %>% filter(uf == curr_uf)
builder = list()
curr_geo

In [27]:
curr_ind = 1

curr_row = curr_geo[curr_ind, ]
curr_geocode = curr_row$geocode
curr_uf = curr_row$uf

geo_req = paste0("&geocode=", curr_geocode, "&uf=", curr_uf)
url = paste0(api, pagination, filters, geo_req)
url

[1] "https://api.mosqlimate.org/api/datastore/climate/?page=1&per_page=200&start=2025-01-01&end=2025-07-05&geocode=1200013&uf=AC"

In [28]:
response = request(url) |> req_headers("X-UID-Key" = api_key) |> req_perform()

In [37]:
resp_items = lapply(resp_body_json(response)$items, as.data.frame)
resp_df = do.call(rbind, resp_items)
resp_df

date,geocodigo,epiweek,temp_min,temp_med,temp_max,precip_min,precip_med,precip_max,precip_tot,pressao_min,pressao_med,pressao_max,umid_min,umid_med,umid_max
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2025-01-01,1200013,202501,23.3049,25.2345,28.7079,0.0000,2.3601,7.6930,18.8811,0.9759,0.9779,0.9804,74.7229,89.8829,97.8283
2025-01-02,1200013,202501,23.2529,25.4844,29.7469,0.0000,1.2815,6.4397,10.2523,0.9752,0.9790,0.9815,68.4804,87.1592,97.5579
2025-01-03,1200013,202501,24.3210,26.0231,29.9635,0.0485,3.8206,14.1589,30.5647,0.9768,0.9790,0.9816,72.4641,89.1319,96.0155
2025-01-04,1200013,202501,23.2695,25.4980,28.7795,0.0000,3.8691,14.2612,30.9529,0.9756,0.9787,0.9810,73.7562,91.4005,99.5058
2025-01-05,1200013,202502,23.6376,25.4265,27.8091,0.4628,3.9595,15.2739,31.6764,0.9764,0.9789,0.9815,84.6081,90.4227,98.0402
2025-01-06,1200013,202502,23.3247,25.9491,30.1018,0.0000,1.5026,5.9834,12.0206,0.9759,0.9786,0.9808,63.7575,86.7761,99.1043
2025-01-07,1200013,202502,22.7550,24.0640,25.2926,0.0000,14.7690,29.2096,118.1518,0.9771,0.9791,0.9816,91.3392,98.1496,99.9974
2025-01-08,1200013,202502,22.1997,24.7181,28.2860,1.0790,6.7835,29.3592,54.2679,0.9748,0.9790,0.9812,72.6622,90.0671,99.9138
2025-01-09,1200013,202502,23.1167,24.9387,27.2293,0.0086,4.2886,9.1956,34.3086,0.9745,0.9779,0.9798,77.2575,93.6908,99.9934


In [ ]:
builder[[curr_geocode]] = resp_df